# 🧪 Taller - Reconocimiento de Acciones Simples con Detección de Postura

Este notebook implementa un sistema en Python que detecta en vivo posturas humanas y reconoce acciones simples como levantar los brazos, sentarse y caminar usando MediaPipe Pose.

## 🎯 Objetivo

Utilizar la webcam para capturar video en tiempo real, detectar puntos clave del cuerpo con MediaPipe Pose y aplicar lógica condicional para reconocer acciones humanas simples.

### 📦 Librerías necesarias

In [23]:
pip install opencv-python mediapipe pygame numpy

In [24]:
import cv2
import mediapipe as mp
import pygame

# Inicializar sonido
pygame.mixer.init()
sound = pygame.mixer.Sound("beep.wav")

# MediaPipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
drawing = mp.solutions.drawing_utils

# Captura de cámara
cap = cv2.VideoCapture(0)

### 🧠 Reconocimiento de acciones simples

Se identifican tres acciones: levantar ambos brazos, estar sentado, y caminar. Además, se reproduce un sonido cuando la persona levanta ambos brazos.

In [25]:
# Variables auxiliares
accion_anterior = None
paso_anterior = None
alternancia = 0
contador_caminata = 0
DURACION_CAMINATA = 30  # frames (~1 segundo)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resultados = pose.process(frame_rgb)
    h, w, _ = frame.shape
    accion = "Sin detectar"

    if resultados.pose_landmarks:
        puntos = resultados.pose_landmarks.landmark

        def y(p): return puntos[p].y * h

        lw_y = y(mp_pose.PoseLandmark.LEFT_WRIST)
        rw_y = y(mp_pose.PoseLandmark.RIGHT_WRIST)
        nose_y = y(mp_pose.PoseLandmark.NOSE)
        lhip_y = y(mp_pose.PoseLandmark.LEFT_HIP)
        rhip_y = y(mp_pose.PoseLandmark.RIGHT_HIP)
        lknee_y = y(mp_pose.PoseLandmark.LEFT_KNEE)
        rknee_y = y(mp_pose.PoseLandmark.RIGHT_KNEE)
        lankle_y = y(mp_pose.PoseLandmark.LEFT_ANKLE)
        rankle_y = y(mp_pose.PoseLandmark.RIGHT_ANKLE)

        # 1️⃣ Brazos arriba
        if lw_y < nose_y and rw_y < nose_y:
            accion = "¡Brazos arriba!"

        # 2️⃣ Sentado
        elif lhip_y > lknee_y and rhip_y > rknee_y:
            accion = "Sentado"

        # 3️⃣ Caminando (con mejor tolerancia)
        else:
            diff_pies = lankle_y - rankle_y

            cambio_signo = paso_anterior is not None and (diff_pies * paso_anterior) < 0
            diferencia_suficiente = paso_anterior is not None and abs(diff_pies - paso_anterior) > 20
            tobillos_no_juntos = abs(diff_pies) > 20

            if cambio_signo and diferencia_suficiente and tobillos_no_juntos:
                alternancia += 1
                contador_caminata = DURACION_CAMINATA

            paso_anterior = diff_pies

            if contador_caminata > 0:
                accion = "Caminando"
                contador_caminata -= 1

        # Dibujar pose
        drawing.draw_landmarks(frame, resultados.pose_landmarks, mp_pose.POSE_CONNECTIONS)

    # Mostrar acción en pantalla
    if accion != accion_anterior and accion != "Sin detectar":
        sound.play()
    accion_anterior = accion

    cv2.putText(frame, accion, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                1, (255, 0, 0), 2)
    cv2.imshow("Reconocimiento de Postura", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()